# Ablation: Frontier Judge Models (Remote APIs)

This notebook compares our local judge models (Llama-3.1-8B, Qwen2.5-14B) against \"Frontier\" models like **GPT-4o**, **Claude 3.5 Sonnet**, and **Gemini 1.5 Pro**.

It supports two remote providers:
1. **OpenRouter**: For OpenAI, Anthropic, DeepSeek, etc. (Requires `OPENROUTER_API_KEY`)
2. **Google GenAI Natively**: For Gemini models (Requires `GEMINI_API_KEY`)

In [ ]:
import warnings, os, logging
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

try:
    import transformers, datasets
    transformers.utils.logging.set_verbosity_error()
    transformers.utils.logging.disable_progress_bar()
    datasets.utils.logging.set_verbosity_error()
    datasets.utils.logging.disable_progress_bar()
    logging.getLogger('sentence_transformers').setLevel(logging.ERROR)
except ImportError: pass

# Ensure required libraries are installed
!uv pip install -e ../..
!uv pip install langchain-openai langchain-google-genai

load_dotenv()
from huggingface_hub import login
login(token=os.getenv('HF_TOKEN'), add_to_git_credential=False)

# Check for API Keys
if not os.getenv("OPENROUTER_API_KEY"):
    print("WARNING: OPENROUTER_API_KEY not found in .env file.")
if not os.getenv("GEMINI_API_KEY"):
    print("WARNING: GEMINI_API_KEY not found in .env file (needed for native Gemini). ")

In [ ]:
from sm_sip.pipelines import run_ablation_experiment

# Models to compare remotely
frontier_judges = [
    "anthropic/claude-3.5-sonnet",
    "openai/gpt-4o-mini",
    "gemini-1.5-pro", # Uses native Google API instead of OpenRouter
    "deepseek/deepseek-chat"
]

run_ablation_experiment(
    presets={'it': ['xlmr-5k-060t'], 'en': ['xlmr-5k-060t']},
    judge_models=frontier_judges,
    num_samples=25, # Reduced samples due to API costs/limits
    ablation_name='judge_remote',
    output_file='results/ablation_judge_remote.json',
)